In [4]:
import os
import typing
from tqdm import tqdm
import glob
import pandas as pd
import tensorflow as tf
import numpy as np
import json

from keras.utils import  Sequence

os.path.exists('/kaggle/input/drive-redused/YaCupTrain') #/kaggle/input/drive-redused/YaCupTrain

2026-02-06 10:25:05.608734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770373505.823510      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770373505.899985      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770373506.483167      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770373506.483217      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770373506.483221      55 computation_placer.cc:177] computation placer alr

True

In [5]:
ROOT_DATA_FOLDER = r"/kaggle/input/drive-redused"

TRAIN_DATASET_PATH = r'/kaggle/input/drive-redused/YaCupTrain'

#os.path.join(ROOT_DATA_FOLDER, r"YaCupTrain")

TEST_DATASET_PATH = os.path.join(ROOT_DATA_FOLDER, r"YaCupTest")
label_columns = ['x', 'y', 'yaw']

# Load all ids of a dataset

def read_testcase_ids(dataset_path: str):
    ids = [int(case_id) for case_id in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, case_id))]
    return ids

ids = read_testcase_ids(TRAIN_DATASET_PATH)
len(ids)

train_ids = np.random.choice(ids, size=round(0.70*len(ids)), replace=False, p=None)
test_ids = [el for el in ids if el not in train_ids]


class DataFilePaths:
    def __init__(self, testcase_path: str):
        self.testcase_path = testcase_path

    def localization(self):
        return os.path.join(self.testcase_path, 'localization.csv')

    def control(self):
        return os.path.join(self.testcase_path, 'control.csv')

    def metadata(self):
        return os.path.join(self.testcase_path, 'metadata.json')

    # exists only for test_dataset
    def requested_stamps(self):
        return os.path.join(self.testcase_path, 'requested_stamps.csv')


def read_localization(localization_path: str):
    return pd.read_csv(localization_path)

def read_control(control_path):
    return pd.read_csv(control_path)

def read_metadata(metadata_path: str):
    with open(metadata_path, 'r') as f:
        data = json.load(f)
    return data

def read_requested_stamps(requested_stamps_path: str):
    return pd.read_csv(requested_stamps_path)

def read_testcase(dataset_path: str, testcase_id: str, is_test: bool = False):
    testcase_path = os.path.join(dataset_path, str(testcase_id))
    data_file_paths = DataFilePaths(testcase_path)

    testcase_data = {}
    testcase_data['localization'] = read_localization(data_file_paths.localization())
    testcase_data['control'] = read_control(data_file_paths.control())
    testcase_data['metadata'] = read_metadata(data_file_paths.metadata())
    if is_test:
        testcase_data['requested_stamps'] = read_requested_stamps(data_file_paths.requested_stamps())

    return testcase_data


def read_testcases(dataset_path: str, is_test: bool = False, testcase_ids: typing.Iterable[int] = None):
    result = {}
    if testcase_ids is None:
        testcase_ids = read_testcase_ids(dataset_path)

    for testcase_id in tqdm(testcase_ids):
        testcase = read_testcase(dataset_path, testcase_id, is_test=is_test)
        result[testcase_id] = testcase
    return result
    
train_dataset = read_testcases(TRAIN_DATASET_PATH)
len(train_dataset)


100%|██████████| 3884/3884 [02:30<00:00, 25.83it/s]


3884

In [21]:
from keras.utils import  Sequence

class FastTrajectoryWindowSequence(Sequence):

    def __init__(
        self,
        cars_data: dict,
        history: int,
        horizon: int,
        batch_size: int,
        stride: int = 1,
        stats: dict | None = None,
        shuffle: bool = True):

        self.history = history
        self.horizon = horizon
        self.batch_size = batch_size
        self.stride = stride
        self.shuffle = shuffle
        self.stats = stats or self._default_stats()
        self._prepare_arrays(cars_data)
        self.on_epoch_end()


    def _default_stats(self):
        return {
            "acc_min": -5.0,
            "acc_max":  5.0,
            "acc_mean": 0.0,
            "acc_std":  1.0,
            "steer_min": -1.5,
            "steer_max":  1.5,
            "steer_mean": 0.0,
            "steer_std":  0.5,}


    def _prepare_arrays(self, cars_data):
        self.data = {}
        self.index = []

        for car_id, car in cars_data.items():
            df = pd.merge_asof(
                car["localization"].sort_values("stamp_ns"),
                car["control"].sort_values("stamp_ns"),
                on="stamp_ns",
                direction="nearest").dropna().reset_index(drop=True)

            # --- numpy arrays ---
            x = df["x"].to_numpy(np.float32)
            y = df["y"].to_numpy(np.float32)
            z = df["z"].to_numpy(np.float32)
            yaw = df["yaw"].to_numpy(np.float32)

            roll = df["roll"].to_numpy(np.float32)
            pitch = df["pitch"].to_numpy(np.float32)

            roll_sin, roll_cos = np.sin(roll), np.cos(roll)
            pitch_sin, pitch_cos = np.sin(pitch), np.cos(pitch)

            acc = np.clip(
                df["acceleration_level"].to_numpy(np.float32),
                self.stats["acc_min"], self.stats["acc_max"])
            steer = np.clip(
                df["steering"].to_numpy(np.float32),
                self.stats["steer_min"], self.stats["steer_max"])

            acc = (acc - self.stats["acc_mean"]) / (self.stats["acc_std"] + 1e-6)
            steer = (steer - self.stats["steer_mean"]) / (self.stats["steer_std"] + 1e-6)

            features = np.stack([
                z, roll_sin, roll_cos,
                pitch_sin, pitch_cos,
                acc, steer], axis=1)

            self.data[car_id] = {
                "features": features,   # (T, 7)
                "xy": np.stack([x, y], axis=1),
                "yaw": yaw}
            T = len(x)
            
            for t0 in range(self.history, T - self.horizon, self.stride):
                self.index.append((car_id, t0))


    def __len__(self):
        return len(self.index) // self.batch_size


    def __getitem__(self, idx):
        batch = self.index[
            idx * self.batch_size : (idx + 1) * self.batch_size]
        X = np.empty((self.batch_size, self.history, 9), np.float32)
        y = np.empty((self.batch_size, self.horizon, 2), np.float32)



        for i, (car_id, t0) in enumerate(batch):

            d = self.data[car_id]
            xy = d["xy"]
            yaw0 = d["yaw"][t0 - 1]

            cos_y, sin_y = np.cos(yaw0), np.sin(yaw0)
            past_xy = xy[t0 - self.history : t0]

            ref = xy[t0 - 1]

            dxy = past_xy - ref
            x_loc =  cos_y * dxy[:, 0] + sin_y * dxy[:, 1]
            y_loc = -sin_y * dxy[:, 0] + cos_y * dxy[:, 1]
            dx = np.diff(np.concatenate([[0.0], x_loc]))
            dy = np.diff(np.concatenate([[0.0], y_loc]))

            X[i] = np.concatenate([
                np.stack([dx, dy], axis=1),
                d["features"][t0 - self.history : t0]
            ], axis=1)
            fut_xy = xy[t0 : t0 + self.horizon] - ref
            x_f =  cos_y * fut_xy[:, 0] + sin_y * fut_xy[:, 1]
            y_f = -sin_y * fut_xy[:, 0] + cos_y * fut_xy[:, 1]
            y[i] = np.stack([x_f, y_f], axis=1)

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.index)

In [22]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_lstm_model(
    history: int,
    horizon: int,
    n_features: int = 9,
    hidden_size: int = 12):
    inputs = layers.Input(shape=(history, n_features))
    x = layers.Masking(mask_value=0.0)(inputs)
    x = layers.LSTM(
       hidden_size,
        return_sequences=False)(x)
    x = layers.RepeatVector(horizon)(x)
    x = layers.LSTM(hidden_size,
        return_sequences=True)(x)
    outputs = layers.TimeDistributed(
        layers.Dense(2)
    )(x)
    model = models.Model(inputs, outputs)
    return model

model = build_lstm_model(20,10)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse")

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 20, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 20, 9)     │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_1 (Masking) │ (None, 20, 9)     │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, 20)        │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 12)        │      1,056 │ masking_1[0][0],  │
│                     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 10, 12)    │          0 │ lstm_2[0][0]      │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 10, 12)    │      1,200 │ repeat_vector_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 10, 2)     │         26 │ lstm_3[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,282 (8.91 KB)

 Trainable params: 2,282 (8.91 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
train_seq = FastTrajectoryWindowSequence(cars_data=train_dataset, history=20,horizon=10,batch_size=32)


In [ ]:
def __init__(
        self,
        cars_data: dict,
        history: int,
        horizon: int,
        batch_size: int,
        stride: int = 1,
        stats: dict | None = None,
        shuffle: bool = True):


In [24]:
model.fit(
    train_seq,
    epochs=3)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3


I0000 00:00:1770374257.224564     125 cuda_dnn.cc:529] Loaded cuDNN version 91002


188296/188296 ━━━━━━━━━━━━━━━━━━━━ 1546s 8ms/step - loss: 0.0213
Epoch 2/3
188296/188296 ━━━━━━━━━━━━━━━━━━━━ 1509s 8ms/step - loss: 5.6577e-04
Epoch 3/3
188296/188296 ━━━━━━━━━━━━━━━━━━━━ 1498s 8ms/step - loss: 4.3224e-04


In [20]:
model.fit(

    train_seq,

    epochs=30,

    #workers=4,

    #use_multiprocessing=True,
    #max_queue_size=10

)

Epoch 1/30


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.